# Telaah: ColPali page-image index (stage 1)

Indexes every page of `bri_2025`, `btn_2025` and `bni_2024` with **vidore/colqwen2-v1.0** and saves
multi-vector embeddings to `/kaggle/working/colpali_index/`, then zips it for download.

**Settings (right panel):** Accelerator = **GPU T4 x2** (not P100: recent PyTorch builds no longer support it),
Internet = **On**, input = your `telaah-pdfs` dataset.

**Run it as a background job:** *Save Version* > *Save & Run All (Commit)*. It keeps running with the browser closed
and the output stays in the version's Output tab.

**Resume after a crash:** add the failed version's output as an extra input. Shards found under
`/kaggle/input/**/colpali_index/` are copied over and skipped.

Page numbers are 1-based PDF pages and match `data/processed/<doc_id>.jsonl` in the local project.

In [ ]:
!pip install -q colpali-engine pymupdf
!pip uninstall -y -q torchao   # Kaggle's old torchao 0.10 breaks peft
import torch, fitz
from importlib.metadata import version
print("torch", torch.__version__, "| transformers", version("transformers"),
      "| colpali-engine", version("colpali-engine"), "| pymupdf", version("pymupdf"))
print("GPU:", torch.cuda.get_device_name(0), "| capability", torch.cuda.get_device_capability(0))

In [ ]:
import glob, os, json, time, shutil
from pathlib import Path

# doc_id -> expected PDF page count (checked locally on 2026-10-13)
DOCS = {"bri_2025": 1407, "btn_2025": 1449, "bni_2024": 1540}
MODEL = "vidore/colqwen2-v1.0"
DPI = 110          # render resolution; the processor resizes again anyway
MAX_SIDE = 1600    # cap the long side of very large pages (px)
BATCH = 4          # pages per forward pass; lower to 2 on out-of-memory
SHARD = 200        # pages per saved shard file
OUT = Path("/kaggle/working/colpali_index")
OUT.mkdir(parents=True, exist_ok=True)

# Find the PDFs wherever Kaggle mounted the dataset
pdfs = {Path(p).stem: p for p in glob.glob("/kaggle/input/**/*.pdf", recursive=True)}
missing = [d for d in DOCS if d not in pdfs]
assert not missing, f"PDFs not found in /kaggle/input: {missing}. Found: {sorted(pdfs)[:10]}"
for d, n in DOCS.items():
    got = fitz.open(pdfs[d]).page_count
    assert got == n, f"{d}: PDF has {got} pages, expected {n}"
    print(f"{d}: {got} pages  ({pdfs[d]})")

# Resume: copy shards from a previous version's output if it was added as input
for p in glob.glob("/kaggle/input/**/colpali_index/*.pt", recursive=True):
    if not (OUT / Path(p).name).exists():
        shutil.copy(p, OUT / Path(p).name)
print("existing shards:", len(list(OUT.glob("*.pt"))))

In [ ]:
from colpali_engine.models import ColQwen2, ColQwen2Processor

# The T4 (compute capability 7.5) has no native bf16, so use fp16 there; bf16 on Ampere or newer
DTYPE = torch.bfloat16 if torch.cuda.get_device_capability(0)[0] >= 8 else torch.float16

def load(dtype):
    m = ColQwen2.from_pretrained(MODEL, torch_dtype=dtype, device_map="cuda:0").eval()
    return m

model = load(DTYPE)
processor = ColQwen2Processor.from_pretrained(MODEL)

def render(doc, page_no):
    pg = doc[page_no - 1]
    zoom = DPI / 72
    w, h = pg.rect.width * zoom, pg.rect.height * zoom
    if max(w, h) > MAX_SIDE:
        zoom *= MAX_SIDE / max(w, h)
    pix = pg.get_pixmap(matrix=fitz.Matrix(zoom, zoom), alpha=False)
    from PIL import Image
    return Image.frombytes("RGB", (pix.width, pix.height), pix.samples)

@torch.inference_mode()
def embed_images(images):
    batch = processor.process_images(images).to(model.device)
    out = model(**batch)                       # (B, L, 128), padded positions are zero
    mask = batch["attention_mask"].bool()
    return [out[i][mask[i]].to(torch.float16).cpu() for i in range(len(images))]

# Sanity check on one real page: fp16 overflow shows up as NaN/inf
with fitz.open(pdfs["bri_2025"]) as d0:
    test = embed_images([render(d0, 328)])[0]
print("test page vectors:", tuple(test.shape), "| finite:", bool(torch.isfinite(test).all()))
if not torch.isfinite(test).all():
    print("fp16 produced NaN/inf -> reloading in float32 (slower)")
    del model; torch.cuda.empty_cache()
    DTYPE = torch.float32
    model = load(DTYPE)
    with fitz.open(pdfs["bri_2025"]) as d0:
        test = embed_images([render(d0, 328)])[0]
    assert torch.isfinite(test).all(), "still not finite in float32"
print("dtype:", DTYPE, "| est. index size:", round(test.numel() * 2 * sum(DOCS.values()) / 1e9, 2), "GB")

In [ ]:
t0 = time.time(); done = 0
total = sum(DOCS.values())
for doc_id, n_pages in DOCS.items():
    doc = fitz.open(pdfs[doc_id])
    for start in range(1, n_pages + 1, SHARD):
        f = OUT / f"{doc_id}_{start:05d}.pt"
        pages = list(range(start, min(start + SHARD, n_pages + 1)))
        if f.exists():
            done += len(pages); continue
        embs = []
        for i in range(0, len(pages), BATCH):
            embs += embed_images([render(doc, p) for p in pages[i:i + BATCH]])
        torch.save({"doc_id": doc_id, "pages": pages,
                    "lengths": [e.shape[0] for e in embs],
                    "emb": torch.cat(embs)}, f)
        done += len(pages)
        el = time.time() - t0
        print(f"{f.name}: {done}/{total} pages | {el/60:.1f} min | ETA {(total-done)*el/max(done,1)/60:.1f} min",
              flush=True)
    doc.close()
print(f"indexing finished in {(time.time()-t0)/60:.1f} min")

In [ ]:
# Manifest: one row per page -> shard file + row offset inside that shard's "emb" tensor
rows = []
for f in sorted(OUT.glob("*.pt")):
    s = torch.load(f)
    off = 0
    for p, n in zip(s["pages"], s["lengths"]):
        rows.append({"doc_id": s["doc_id"], "page": p, "shard": f.name, "offset": off, "n_vectors": n})
        off += n
assert len(rows) == sum(DOCS.values()), f"{len(rows)} pages in shards, expected {sum(DOCS.values())}"
meta = {"model": MODEL, "dtype_model": str(DTYPE), "dtype_saved": "float16", "dpi": DPI,
        "max_side": MAX_SIDE, "dim": 128, "docs": DOCS, "pages": rows}
(OUT / "manifest.json").write_text(json.dumps(meta))
print(len(rows), "pages | avg vectors/page:", round(sum(r["n_vectors"] for r in rows) / len(rows), 1))

In [ ]:
# Quick retrieval test on the chart questions (q028-q032) before downloading anything
QS = [
    ("q028", "Menurut grafik Laporan Laba Rugi BRI 2025, berapa laba operasional BRI tahun 2025 (dalam triliun Rupiah)?", ("bri_2025", 328)),
    ("q029", "Menurut grafik Laporan Posisi Keuangan Konsolidasian BRI 2025, berapa total ekuitas BRI tahun 2025 (dalam triliun Rupiah)?", ("bri_2025", 315)),
    ("q030", "Menurut grafik pertumbuhan kredit dan DPK perbankan dalam laporan tahunan BTN 2025, berapa LDR industri perbankan pada Desember 2025?", ("btn_2025", 248)),
    ("q031", "Menurut grafik pertumbuhan kredit dan DPK perbankan dalam laporan tahunan BTN 2025, berapa pertumbuhan DPK perbankan (yoy) pada Januari 2024?", ("btn_2025", 248)),
    ("q032", "Menurut grafik perbandingan harga saham BBNI dan IHSG pada laporan tahunan BNI 2024, berapa nilai IHSG pada Desember 2024?", ("bni_2024", 34)),
]
page_embs, keys = [], []
for f in sorted(OUT.glob("*.pt")):
    s = torch.load(f)
    page_embs += list(torch.split(s["emb"], s["lengths"]))
    keys += [(s["doc_id"], p) for p in s["pages"]]

with torch.inference_mode():
    qb = processor.process_queries([q for _, q, _ in QS]).to(model.device)
    q_embs = [e.to(torch.float16).cpu() for e in model(**qb)]
scores = processor.score_multi_vector(q_embs, page_embs, batch_size=256, device="cuda:0")  # (Q, P)

def run(only_doc):
    hits = 0
    for i, (qid, _, gold) in enumerate(QS):
        sc = scores[i].clone()
        if only_doc:  # mimic the local doc filter: restrict to the question's report
            sc[[j for j, k in enumerate(keys) if k[0] != gold[0]]] = float("-inf")
        top = [keys[j] for j in sc.topk(5).indices.tolist()]
        hit = gold in top; hits += hit
        print(f"  {qid} gold {gold} {'HIT ' if hit else 'miss'} top5={[f'{d}:{p}' for d, p in top]}")
    print(f"  recall@5 = {hits}/{len(QS)}")

print("All 3 reports:"); run(False)
print("Filtered to the question's report:"); run(True)

In [ ]:
shutil.make_archive("/kaggle/working/colpali_index", "zip", OUT)
print(round(os.path.getsize("/kaggle/working/colpali_index.zip") / 1e9, 2), "GB -> download colpali_index.zip from the Output tab")